In [1]:
# Repo paths 
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / 'data').exists():
            return p
    return start

ROOT = find_repo_root()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PLOTS = ROOT / 'plots'
PLOTS.mkdir(exist_ok=True)

# Imports

In [2]:
import pandas as pd
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

path = ROOT / 'chatlogs.csv'
df = pd.read_csv(path)

print(df.head())

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


   Unnamed: 0              message association_to_offender      time  \
0           0           gold 2 zed                   enemy  00:00:21   
1           1                 IIII                   enemy  00:00:27   
2           2  nice premade lie :o                   enemy  00:00:27   
3           3                  ISI                   enemy  00:00:28   
4           4        smiteless pls                   enemy  00:00:43   

   case_total_reports  allied_report_count  enemy_report_count  \
0                   8                    0                   2   
1                   8                    0                   2   
2                   8                    0                   2   
3                   8                    0                   2   
4                   8                    0                   2   

  most_common_report_reason  chatlog_id champion_name  
0         Negative Attitude           1          Udyr  
1         Negative Attitude           1         Riven  
2 

# Text Cleaning


In [3]:
df['text']=df['message']

In [4]:
df.head()

,Unnamed: 0,message,association_to_offender,time,case_total_reports,allied_report_count,enemy_report_count,most_common_report_reason,chatlog_id,champion_name,text
0,0,gold 2 zed,enemy,00:00:21,8,0,2,Negative Attitude,1,Udyr,gold 2 zed
1,1,IIII,enemy,00:00:27,8,0,2,Negative Attitude,1,Riven,IIII
2,2,nice premade lie :o,enemy,00:00:27,8,0,2,Negative Attitude,1,Udyr,nice premade lie :o
3,3,ISI,enemy,00:00:28,8,0,2,Negative Attitude,1,Riven,ISI
4,4,smiteless pls,enemy,00:00:43,8,0,2,Negative Attitude,1,Udyr,smiteless pls


# Tokenization & Padding

In [10]:
# 1. Handle missing values: Replace NaNs with empty strings to prevent errors during text concatenation
df['message'] = df['message'].fillna('')

# 2. Group data by match and player
# Concatenate all messages from the same player in a specific match, separated by a space.
# We also retain the 'association_to_offender' status, which is constant for a given player per match.
grouped_df = df.groupby(['chatlog_id', 'champion_name']).agg({
    'message': lambda x: ' '.join(x),
    'association_to_offender': 'first'
}).reset_index()

# 3. Create the binary target column
# The player convicted in the tribunal (offender) is labeled 1 (toxic), all others are 0
grouped_df['is_toxic'] = (grouped_df['association_to_offender'] == 'offender').astype(int)

# 4. Display results to verify the new structure
print("Shape of the new aggregated dataset:", grouped_df.shape)
print("\nSample rows:")
print(grouped_df[['chatlog_id', 'champion_name', 'message', 'is_toxic']].head())

# Check label distribution (count of toxic vs. non-toxic samples)
print("\nLabel distribution:")
print(grouped_df['is_toxic'].value_counts())

Shape of the new aggregated dataset: (88088, 5)

Sample rows:
   chatlog_id champion_name  \
0           1        Ezreal   
1           1         Janna   
2           1          Jinx   
3           1         Karma   
4           1         Riven   

                                             message  is_toxic  
0  report for unskilled player is useless thx <3 ...         0  
1                                             mimimi         0  
2  im comming for you riven pfft focus Zed always...         0  
3  thx top no flash for what ? he has 2 kill in l...         0  
4  IIII ISI K udyr top dnt us see it? CAMP MORE P...         0  

Label distribution:
is_toxic
0    78368
1     9720
Name: count, dtype: int64


In [11]:



def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'[^a-z0-9\s]', '', text) # deleting chars that are not letters or numbers combination
    text = re.sub(r'\s+', ' ', text).strip() # deleting double spaces

    return text

# applying the clean_text function on the grouped messages
grouped_df['cleaned_text'] = grouped_df['message'].apply(clean_text)

MAX_WORDS = 10000  # vocabulary of 10,000 words
MAX_LEN = 300      # max of 300 words per game (increased from 50 because messages are grouped)

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(grouped_df['cleaned_text'])

sequences = tokenizer.texts_to_sequences(grouped_df['cleaned_text'])

# sentences < 300 words will get padding of zeros to fit 300 words and sentences > 300 words will be cut
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post') 

# getting the target labels (1 for offender, 0 for others)
y = grouped_df['is_toxic'].values

print(grouped_df['is_toxic'].value_counts())

print("Shape of data tensor (X):", X.shape)
print("Shape of label tensor (y):", y.shape)

is_toxic
0    78368
1     9720
Name: count, dtype: int64
Shape of data tensor (X): (88088, 300)
Shape of label tensor (y): (88088,)
